# rayforge-cuda: RayForge ported to the GPU

Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.


In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!nvcc --version | tail -2
!nproc

In [ ]:
!git clone -q https://github.com/aarxa/rayforge-cuda
!git clone -q --depth 1 https://github.com/aarxa/RayForge-CPP-Ray-Tracing-Engine
!ls -la RayForge-CPP-Ray-Tracing-Engine/dragon.ray RayForge-CPP-Ray-Tracing-Engine/buddha.ray

In [ ]:
%cd rayforge-cuda
!make rayforge_gpu rayforge_cpu ARCH=native

In [ ]:
# RayForge's own CPU renders (1920x1080), stored as PNG; convert to BMP for the comparison
from PIL import Image
for s in ['dragon','buddha']:
    Image.open(f'reference/{s}_cpu_1080p.png').convert('RGB').save(f'reference/{s}_cpu_1080p.bmp')

## 1080p renders, checked against the CPU build and against original RayForge

In [ ]:
!./rayforge_gpu --in ../RayForge-CPP-Ray-Tracing-Engine/dragon.ray --ref reference/dragon_cpu_1080p.bmp --cpu-check --out dragon_gpu.bmp --reps 10

In [ ]:
!./rayforge_gpu --in ../RayForge-CPP-Ray-Tracing-Engine/buddha.ray --ref reference/buddha_cpu_1080p.bmp --cpu-check --out buddha_gpu.bmp --reps 10

## Ablations: what each design choice is worth

In [ ]:
for s in ['dragon','buddha']:
    print('=== '+s+': median-split BVH ===')
    !./rayforge_gpu --in ../RayForge-CPP-Ray-Tracing-Engine/$s.ray --bvh median --reps 10 | grep -E 'BVH|GPU render'
    print('=== '+s+': closest-hit shadow rays ===')
    !./rayforge_gpu --in ../RayForge-CPP-Ray-Tracing-Engine/$s.ray --shadows closest --reps 10 | grep 'GPU render'

## 4K

In [ ]:
!./rayforge_gpu --in ../RayForge-CPP-Ray-Tracing-Engine/buddha.ray --width 3840 --height 2160 --reps 10 --out buddha_4k_gpu.bmp | grep -E 'GPU render'

In [ ]:
from IPython.display import display
for f in ['dragon_gpu.bmp','buddha_gpu.bmp']:
    im=Image.open(f); im.thumbnail((960,540)); display(im)